In [83]:
%load_ext autoreload
%autoreload 2
%env CUDA_VISIBLE_DEVICES=1

import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import vis_utils

from utils.utils import get_path
from utils.io_utils import load_multiple_res, read_ripser_result
from utils.toydata_utils import get_toy_data
from utils.fig_utils import dataset_to_print, plot_dgm_loops, dist_to_print, plot_edges_on_scatter
from utils.pd_utils import sort_cycle
from utils.confidence_utils import median_max_life_time, get_bottleneck_dist
from utils.passing_cells_utils import distribution_max, distribution_split

from vis_utils.loaders import load_dataset
from vis_utils.plot import plot_scatter
from vis_utils.utils import load_dict, save_dict
from vis_utils.tsne_wrapper import TSNEwrapper
from vis_utils.confidence import fig_bottleneck, fig_bottleneck_max, fig_bottleneck_median, get_life_geq_median, get_life_geq_metrics

from openTSNE.affinity import Affinities

import umap
from ripser import Rips
from ripser.ripser import get_greedy_perm

from mpl_toolkits.axes_grid1 import make_axes_locatable

from matplotlib import collections  as mc
from matplotlib.colors import Normalize
import matplotlib.cm

from openTSNE.nearest_neighbors import PrecomputedNeighbors
from openTSNE.affinity import PerplexityBasedNN

from vis_utils.utils import kNN_graph, kNN_dists
from scipy.spatial.distance import pdist, squareform
import glasbey
import scipy.sparse
import networkx as nx

from sklearn.decomposition import PCA

from persim import plot_diagrams
from utils.pd_utils import get_life_times
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUDA_VISIBLE_DEVICES=1


In [84]:
root_path = get_path("data")
fig_path = get_path("figures")
seeds = [0]
k = 15

style_file = "utils.style"
plt.style.use(style_file)

distances = {
    # "euclidean": [{}],
    "diffusion": [
        {"k": 15, "t": 8, "kernel": "sknn", "include_self": False},
        {"k": 100, "t": 8, "kernel": "sknn", "include_self": False},
        {"k": 15, "t": 64, "kernel": "sknn", "include_self": False},
        {"k": 100, "t": 64, "kernel": "sknn", "include_self": False},
    ],
    "eff_res": [
        {"corrected": True, "weighted": False, "k": 15, "disconnect": True},
        {"corrected": True, "weighted": False, "k": 100, "disconnect": True},
    ],
}

## embedding loading and mask making 

In [ ]:
sex_color = "male_orange"
for i in range(10):
    globals()[f"embd_10x_{sex_color}_{i}"] = np.load(os.path.join(root_path, "embd_n_mask","embd_after_procrustes", f"embd_10x_{sex_color}_split_{i}.npy"))
    print(globals()[f"embd_10x_{sex_color}_{i}"].shape)

(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5288, 2)


In [ ]:
sex_color = "female_orange"
for i in range(5):
    globals()[f"embd_10x_{sex_color}_{i}"] = np.load(os.path.join(root_path, "embd_n_mask","embd_after_procrustes", f"embd_10x_{sex_color}_split_{i}.npy"))
    print(globals()[f"embd_10x_{sex_color}_{i}"].shape)

(5000, 2)
(5000, 2)
(5000, 2)
(5000, 2)
(5000, 2)


# get_life_geq_median (2/3 * median of 25 resample) where method ==  split

## scale = 2

In [ ]:
_, _, _, _, full_d_male = load_dataset(root_path, "yao_10x_male", k=k)
cluster_orange_male = [cluster for i, cluster in enumerate(full_d_male["clusterNames"]) 
                  if "_Sst" in cluster or "_Pvalb" in cluster]
idx_orange = [i for i, cluster in enumerate(full_d_male["clusterNames"]) 
              if "_Sst" in cluster or "_Pvalb" in cluster]
mask_orange_male = [idx in idx_orange for idx in full_d_male['clusters']]

In [ ]:
_, _, _, _, full_d_female = load_dataset(root_path, "yao_10x_female", k=k)
cluster_orange_female = [cluster for i, cluster in enumerate(full_d_female["clusterNames"]) 
                  if "_Sst" in cluster or "_Pvalb" in cluster]
idx_orange = [i for i, cluster in enumerate(full_d_female["clusterNames"]) 
              if "_Sst" in cluster or "_Pvalb" in cluster]
mask_orange_female = [idx in idx_orange for idx in full_d_female['clusters']]

In [ ]:
for i in range(5):
    dataset = f"yao_10x_female_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    life_maxs, geq_maxs = get_life_geq_median(all_res, dataset, method="split", ratio = "quarter", scale = 2)
    print(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_female, full_d = full_d_female, mask = mask_orange_female, geq_maxs = geq_maxs, total_num = 5)
    # save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_scale_row", f"dist_10x_female_orange_split_{i}_scale_2.pkl"))

Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[7, 2, 2, 2, 5, 4]
Done with yao_10x_female_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_1 None diffusion_k_100_t_8_kern

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(5) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(5) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_scale_row", f"dist_10x_female_orange_splits_scale_2.pkl"))

In [ ]:
for i in range(10):
    dataset = f"yao_10x_male_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    life_maxs, geq_maxs = get_life_geq_median(all_res, dataset, method="split", ratio = "quarter", scale = 2)
    print(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_male, full_d = full_d_male, mask = mask_orange_male, geq_maxs = geq_maxs, total_num = 10)
    # save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_scale_row", f"dist_10x_male_orange_split_{i}_scale_2.pkl"))

Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[7, 3, 3, 2, 1, 7]
Done with yao_10x_male_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_100_t_8_kernel_sknn_include_

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(10) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(10) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_scale_row", f"dist_10x_male_orange_splits_scale_2.pkl"))

## scale = 3

In [ ]:
for i in range(5):
    dataset = f"yao_10x_female_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    life_maxs, geq_maxs = get_life_geq_median(all_res, dataset, method="split", ratio = "quarter", scale = 3)
    print(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_female, full_d = full_d_female, mask = mask_orange_female, geq_maxs = geq_maxs, total_num = 5)
    # save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_scale_row", f"dist_10x_female_orange_split_{i}_scale_3.pkl"))

Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[0, 2, 0, 0, 0, 0]
Done with yao_10x_female_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_1 None diffusion_k_100_t_8_kern

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(5) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(5) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_scale_row", f"dist_10x_female_orange_splits_scale_3.pkl"))

In [ ]:
for i in range(10):
    dataset = f"yao_10x_male_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    life_maxs, geq_maxs = get_life_geq_median(all_res, dataset, method="split", ratio = "quarter", scale = 3)
    print(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_male, full_d = full_d_male, mask = mask_orange_male, geq_maxs = geq_maxs, total_num = 10)
    # save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_scale_row", f"dist_10x_male_orange_split_{i}_scale_3.pkl"))

Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[2, 2, 1, 0, 0, 1]
Done with yao_10x_male_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_100_t_8_kernel_sknn_include_

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(10) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(10) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_scale_row", f"dist_10x_male_orange_splits_scale_3.pkl"))

# get_life_geq_metrics(2 * (std + median) or 2 * (std + mean) of 25 resample) where method ==  split

## mean

In [ ]:
geq_maxss = []

In [ ]:
sex_data = [f"yao_10x_male_orange_split_{i}" for i in range(10)]+ [f"yao_10x_female_orange_split_{i}" for i in range(5)]

In [ ]:
for dataset in sex_data:
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    _, geq_maxs = get_life_geq_metrics(all_res, dataset, method = "split", ratio = "quarter", scale = 2, metrics = "mean")
    geq_maxss.append(geq_maxs)

Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outlie

In [ ]:
geq_maxss

[[2, 2, 1, 1, 0, 1],
 [2, 2, 1, 0, 1, 2],
 [2, 2, 3, 0, 0, 3],
 [3, 3, 2, 1, 2, 2],
 [1, 2, 1, 0, 0, 3],
 [1, 2, 1, 2, 0, 2],
 [2, 2, 1, 0, 0, 3],
 [2, 1, 1, 2, 1, 2],
 [2, 3, 1, 0, 2, 2],
 [2, 2, 0, 1, 2, 2],
 [0, 2, 0, 0, 0, 2],
 [1, 3, 1, 1, 1, 2],
 [0, 0, 0, 1, 0, 0],
 [2, 1, 0, 0, 0, 1],
 [0, 1, 0, 1, 0, 1]]

In [ ]:
np.array(geq_maxss)[10:].sum()

21

In [ ]:
sum([22, 28, 13, 10,  9, 28])

110

## median

In [ ]:
geq_maxss = []

In [ ]:
for dataset in sex_data:
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    _, geq_maxs = get_life_geq_metrics(all_res, dataset, method = "split", ratio = "quarter", scale = 2, metrics = "median")
    geq_maxss.append(geq_maxs)

Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outlie

In [ ]:
np.array(geq_maxss)[10:].sum(0)

array([4, 7, 1, 6, 6, 5])

# check the bottleneck distance to itself....

conclusion: its no 0 since i set the threshold = "quarter", and it takes features whose lifetime is in top 25%. the bottleneck distance is taken as max{the shortest lifetime, the shortest distance between matching dots on the diagram}, when calculating the bottleneck distance of a PD to itself, the later term is 0, but the first term is not, but a small number that is represented in the final result.

In [ ]:
sex_data = [f"yao_10x_male_orange_split_{i}" for i in range(10)]+ [f"yao_10x_female_orange_split_{i}" for i in range(5)]


In [ ]:
full_dists = []
for i, distance in enumerate(distances):
        for j, full_dist in enumerate(all_res[distance]):
            full_dists.append(full_dist)

In [ ]:
for full_dist in full_dists: 
    dgms = []
    for com_dataset in sex_data:
        file_name = os.path.join(root_path,
                                com_dataset,
                                f"{com_dataset}_seed_0_{full_dist}_rep")
        dgms.append(read_ripser_result(file_name))
    for i, dataset in enumerate(sex_data):
        reference_dgm = dgms[i]
        bdist = load_dict(os.path.join(get_path("data"),f"{dataset}/bottleneck_dists_split_quarter_{full_dist}.pkl"))["bottleneck"]
        threshold = int(np.floor(len(reference_dgm["dgms"][1])/4))
        filtered_dgms, max_del_lts = zip(*[filter_diagram(dgm, dim=1, n=threshold)
                                        for dgm in dgms])
        if max(max_del_lts) != min(bdist):
            print(dataset, full_dist, max(max_del_lts), min(bdist))

# check is male and female splits have different bdist distribution(means or smth)

In [ ]:
from scipy.stats import ttest_ind

In [ ]:
var = True
# print("t-test: [ttest_ind(mm, ff), ttest_ind(mm, mf), ttest_ind(ff, mf]")
print("[np.mean(mm), np.mean(ff), np.mean(mf)]")
print("------------------------------------------------------")
for full_dist in full_dists:
    bdists = [] 
    for i, dataset in enumerate(sex_data):
        bdist = load_dict(os.path.join(get_path("data"),f"{dataset}/bottleneck_dists_split_quarter_{full_dist}.pkl"))["bottleneck"]
        bdists.append(bdist)
    bdists = np.array(bdists)
    mm = bdists[:10,:10][np.triu_indices_from(bdists[:10,:10], k=1)]
    ff = bdists[10:,10:][np.triu_indices_from(bdists[10:,10:], k=1)]
    mf = bdists[:10,10:].flatten()
    t_stat, p_0 = ttest_ind(mm, ff, equal_var=var)
    t_stat, p_1 = ttest_ind(mm, mf, equal_var=var)
    t_stat, p_2 = ttest_ind(ff, mf, equal_var=var)
    
    ps  = np.hstack([p_0,p_1,p_2])
    means = np.hstack([np.mean(mm), np.mean(ff), np.mean(mf)])
    np.set_printoptions(precision=6, suppress=True)
    # print(full_dist, ps)
    print(full_dist, means)

[np.mean(mm), np.mean(ff), np.mean(mf)]
------------------------------------------------------
diffusion_k_15_t_8_kernel_sknn_include_self_False [0.219163 0.275715 0.239614]
diffusion_k_100_t_8_kernel_sknn_include_self_False [0.104063 0.098019 0.110414]
diffusion_k_15_t_64_kernel_sknn_include_self_False [0.029671 0.035214 0.036808]
diffusion_k_100_t_64_kernel_sknn_include_self_False [0.006487 0.005568 0.006127]
eff_res_corrected_True_weighted_False_k_15_disconnect_True [0.001207 0.000829 0.001057]
eff_res_corrected_True_weighted_False_k_100_disconnect_True [0.000022 0.000018 0.000022]


In [ ]:
bdists = np.array(bdists)

In [ ]:
bdists.shape

(90, 15)

# Does female data tend to have smaller lifetime????

## yes and no

In [ ]:
test_num = 3
for full_dist in full_dists: 
    means = []
    for com_dataset in sex_data:
        file_name = os.path.join(root_path,
                                com_dataset,
                                f"{com_dataset}_seed_0_{full_dist}_rep")
        res = read_ripser_result(file_name)        
        life_times_loops = get_life_times(res, dim=1)
        loop_idx_sorted = sorted(life_times_loops, reverse = True)[:test_num]
        means.append(np.median(loop_idx_sorted))
    np.set_printoptions(precision=6, suppress=True)
    print(np.median(means[:10]),np.median(means[10:]))
        

0.5917014999999999 0.571463
0.31320249999999994 0.25654299999999997
0.06244854999999999 0.06372159999999999
0.012795049999999999 0.013517
0.0025394550000000004 0.0023257999999999994
6.699035e-05 4.53483e-05


In [ ]:
test_num = 3
for full_dist in full_dists: 
    means = []
    for com_dataset in sex_data:
        file_name = os.path.join(root_path,
                                com_dataset,
                                f"{com_dataset}_seed_0_{full_dist}_rep")
        res = read_ripser_result(file_name)        
        life_times_loops = get_life_times(res, dim=1)
        loop_idx_sorted = sorted(life_times_loops, reverse = True)[:test_num]
        means.append(np.mean(loop_idx_sorted))
    np.set_printoptions(precision=6, suppress=True)
    print(np.mean(means[:10]),np.mean(means[10:]))

0.6099978666666667 0.5703229333333334
0.3191312666666667 0.2729776666666667
0.07804793333333333 0.06692716000000001
0.012977286999999999 0.013187925999999999
0.002529911 0.0022710386666666667
6.400162666666666e-05 4.645481333333334e-05
